# 将非结构化数据从本地文件转换为 ClickZetta Lakehouse 中的 RAG 就绪数据

## 概述
本 Notebook 演示了如何使用 unstructured-ingest-clickzetta 工具链将本地 Markdown 文档处理成适合 RAG（检索增强生成）应用的向量化数据，并存储到 ClickZetta Lakehouse 中。

### 主要特性
- 🚀 **高性能 ETL**: 使用 unstructured.io 的并行处理能力
- 🧠 **智能嵌入**: 集成阿里云 DashScope text-embedding-v4 模型 (1024维)
- 💾 **向量数据库**: ClickZetta Lakehouse 原生向量支持
- 🔍 **语义检索**: 基于余弦距离的向量相似度搜索
- 🛠️ **完整工具链**: 从数据摄取到查询的端到端解决方案

### 处理流程
1. **数据摄取**: 扫描本地 Markdown 文件
2. **文档解析**: 使用 unstructured.io 解析文档结构
3. **智能分块**: 按标题层次进行语义分块
4. **向量化**: 使用 DashScope 生成 1024 维向量
5. **数据存储**: 存储到 ClickZetta 的 Raw 和 Silver 表
6. **语义检索**: 支持自然语言查询和向量检索

## 📋 环境要求

确保已安装以下依赖：
- Python 3.8+
- ClickZetta 连接器
- DashScope SDK
- unstructured-ingest-clickzetta（本项目）

配置 .env 文件中的必需环境变量：
```bash
# ClickZetta 数据库配置
CLICKZETTA_SERVICE=your-service
CLICKZETTA_INSTANCE=your-instance
CLICKZETTA_WORKSPACE=your-workspace
CLICKZETTA_SCHEMA=your-schema
CLICKZETTA_USERNAME=your-username
CLICKZETTA_PASSWORD=your-password
CLICKZETTA_VCLUSTER=your-vcluster

# DashScope AI 模型配置
DASHSCOPE_API_KEY=your-api-key

# 本地文件路径
LOCAL_FILE_INPUT_DIR=/path/to/your/documents
```

In [1]:
# 🔧 第1步：环境准备和依赖安装

import sys
import importlib

print("📦 安装unstructured-ingest-clickzetta包...")

# 方式1: 从PyPI安装（推荐）
!pip install unstructured-ingest-clickzetta --upgrade -q

# 方式2: 从GitHub安装最新开发版本
# !pip install git+https://github.com/yunqiqiliang/unstructured-ingest-clickzetta.git@v1.3.1 -q

# 方式3: 本地开发版本（仅用于开发调试）
# !pip uninstall unstructured-ingest-clickzetta -y -q
# !pip install -e /Users/liangmo/Documents/GitHub/unstructured-ingest-clickzetta/ -q

# 清理模块缓存，强制重新导入
modules_to_remove = [module for module in sys.modules.keys() if module.startswith('unstructured_ingest')]
for module in modules_to_remove:
    if module in sys.modules:
        del sys.modules[module]

# 验证安装
try:
    import unstructured_ingest
    # 从正确的模块导入版本信息
    from unstructured_ingest import __version__
    print(f"✅ 成功安装包版本: {__version__}")
except ImportError as e:
    print(f"❌ 导入失败: {e}")
except AttributeError:
    # 如果__version__不存在，尝试其他方式获取版本
    try:
        import pkg_resources
        version = pkg_resources.get_distribution("unstructured-ingest-clickzetta").version
        print(f"✅ 成功安装包版本: {version}")
    except:
        print("✅ 包已成功导入（版本信息不可用）")

# 验证 DashScope 支持
try:
    from unstructured_ingest.processes.embedder import EmbedderConfig
    test_config = EmbedderConfig(
        embedding_provider="dashscope",
        embedding_model_name="text-embedding-v4",
        embedding_api_key="test"
    )
    print("✅ DashScope 嵌入器支持正常")
    
    # 测试 get_embedder 方法
    embedder = test_config.get_embedder()
    print("✅ DashScope 嵌入器初始化成功")
        
except Exception as e:
    print(f"❌ DashScope 支持检查失败: {e}")
    import traceback
    traceback.print_exc()

print("\n🎯 最新版本特性:")
print("  ✓ 优化了代码结构，减少重复逻辑")
print("  ✓ 提升了性能，静默调试输出")  
print("  ✓ 增强了列处理的可维护性")
print("  ✓ 符合DRY原则的代码重构")

print("\n💡 安装方式说明:")
print("  📦 从PyPI安装: pip install unstructured-ingest-clickzetta")
print("  🌐 从GitHub安装: pip install git+https://github.com/yunqiqiliang/unstructured-ingest-clickzetta.git")
print("  🔧 本地开发: pip install -e .")

📦 安装unstructured-ingest-clickzetta包...
✅ 成功安装包版本: <module 'unstructured_ingest.__version__' from '/Users/liangmo/anaconda3/envs/unstructured311/lib/python3.11/site-packages/unstructured_ingest/__version__.py'>
✅ DashScope 嵌入器支持正常
✅ DashScope 嵌入器初始化成功

🎯 最新版本特性:
  ✓ 优化了代码结构，减少重复逻辑
  ✓ 提升了性能，静默调试输出
  ✓ 增强了列处理的可维护性
  ✓ 符合DRY原则的代码重构

💡 安装方式说明:
  📦 从PyPI安装: pip install unstructured-ingest-clickzetta
  🌐 从GitHub安装: pip install git+https://github.com/yunqiqiliang/unstructured-ingest-clickzetta.git
  🔧 本地开发: pip install -e .


In [2]:
# 📦 第2步：导入必需的库和配置日志

import json
import pandas as pd
import logging
import warnings

# 简化的日志配置 - 只显示ERROR级别
logging.basicConfig(level=logging.ERROR, force=True)
warnings.filterwarnings("ignore")

# 配置参数
DROP_TABLES_BEFORE_RUN = True  # 是否在运行前删除现有表
ENABLE_DEBUG_MODE = False      # 关闭调试模式

print("📋 生产模式配置完成")
print(f"⚙️  配置: {'重建表' if DROP_TABLES_BEFORE_RUN else '追加数据'}")
print("🔇 日志级别: ERROR（静默模式）")

📋 生产模式配置完成
⚙️  配置: 重建表
🔇 日志级别: ERROR（静默模式）


In [3]:
# ⚙️ 第3步：核心配置参数

# 表命名和前缀配置 - 恢复到之前工作的配置
INDEX_AND_TABLE_PREFIX = "dashscope_v4_1024_2048_20250611_"  # 使用之前工作的前缀
RAW_TABLE_NAME = f"{INDEX_AND_TABLE_PREFIX}yunqi_raw_elements"
SILVER_TABLE_NAME = f"{INDEX_AND_TABLE_PREFIX}yunqi_elements"

# DashScope 嵌入模型配置
EMBEDDING_PROVIDER = "dashscope"
EMBEDDING_MODEL_NAME = "text-embedding-v4"  # DashScope 最新的 v4 模型
EMBEDDINGS_DIMENSIONS = 1024                 # text-embedding-v4 的向量维度

# 文本分块配置
CHUNK_MAX_CHARACTERS = 2048  # 单个文本块的最大字符数
CHUNK_OVERLAP = 512          # 文本块之间的重叠字符数
CHUNK_COMBINE_THRESHOLD = 200 # 小于此字符数的块会被合并

# 文档来源标识
DOCUMENTS_SOURCE = "https://yunqi.tech/documents"

print("✅ 配置参数已加载（恢复到工作状态）:")
print(f"  📊 数据表前缀: {INDEX_AND_TABLE_PREFIX}")
print(f"  🧠 嵌入模型: {EMBEDDING_MODEL_NAME}")
print(f"  📏 向量维度: {EMBEDDINGS_DIMENSIONS}")
print(f"  📝 文本块大小: {CHUNK_MAX_CHARACTERS} 字符 (重叠: {CHUNK_OVERLAP})")
print(f"  🏷️  数据来源: {DOCUMENTS_SOURCE}")

✅ 配置参数已加载（恢复到工作状态）:
  📊 数据表前缀: dashscope_v4_1024_2048_20250611_
  🧠 嵌入模型: text-embedding-v4
  📏 向量维度: 1024
  📝 文本块大小: 2048 字符 (重叠: 512)
  🏷️  数据来源: https://yunqi.tech/documents


In [4]:
# 🔐 第4步：环境变量加载和验证

import os
from pathlib import Path
from dotenv import load_dotenv

def load_environment_variables():
    """加载并验证环境变量"""
    # 加载项目根目录下的 .env 文件
    project_root = Path(__file__).parent.parent.parent if '__file__' in globals() else Path.cwd().parent.parent
    env_path = project_root / '.env'
    
    if env_path.exists():
        load_dotenv(env_path)
        print(f"✅ 已加载环境变量文件: {env_path}")
    else:
        print(f"⚠️  未找到 .env 文件: {env_path}")
        print("将尝试使用系统环境变量")
    
    # 获取环境变量
    config = {
        'username': os.getenv("CLICKZETTA_USERNAME"),
        'password': os.getenv("CLICKZETTA_PASSWORD"),
        'service': os.getenv("CLICKZETTA_SERVICE"),
        'instance': os.getenv("CLICKZETTA_INSTANCE"),
        'workspace': os.getenv("CLICKZETTA_WORKSPACE"),
        'schema': os.getenv("CLICKZETTA_SCHEMA"),
        'vcluster': os.getenv("CLICKZETTA_VCLUSTER"),
        'api_key': os.getenv("DASHSCOPE_API_KEY"),
        'connection_timeout': os.getenv("CLICKZETTA_CONNECTION_TIMEOUT", "60"),
        'query_timeout': os.getenv("CLICKZETTA_QUERY_TIMEOUT", "1800"),
        'input_dir': os.getenv("LOCAL_FILE_INPUT_DIR")
    }
    
    # 验证必需参数
    required_params = [
        ("CLICKZETTA_USERNAME", config['username']),
        ("CLICKZETTA_PASSWORD", config['password']),
        ("CLICKZETTA_SERVICE", config['service']),
        ("CLICKZETTA_INSTANCE", config['instance']),
        ("CLICKZETTA_WORKSPACE", config['workspace']),
        ("CLICKZETTA_SCHEMA", config['schema']),
        ("CLICKZETTA_VCLUSTER", config['vcluster']),
        ("DASHSCOPE_API_KEY", config['api_key']),
        ("LOCAL_FILE_INPUT_DIR", config['input_dir'])
    ]
    
    missing_params = [name for name, value in required_params if not value]
    if missing_params:
        print(f"❌ 缺少必需的环境变量: {', '.join(missing_params)}")
        print("请确保在项目根目录的 .env 文件中设置了所有必需的环境变量")
        raise ValueError(f"Missing required environment variables: {missing_params}")
    
    print("✅ 所有必需的环境变量已成功加载")
    print(f"🌐 连接目标: {config['service']} (实例: {config['instance']})")
    print(f"📁 工作空间: {config['workspace']}.{config['schema']}")
    print(f"⚡ 虚拟集群: {config['vcluster']}")
    print(f"📂 输入目录: {config['input_dir']}")
    
    return config

# 加载配置
try:
    CONFIG = load_environment_variables()
    print("\n🎉 环境配置验证成功！")
except Exception as e:
    print(f"\n❌ 环境配置失败: {e}")
    raise

✅ 已加载环境变量文件: /Users/liangmo/Documents/GitHub/unstructured-ingest-clickzetta/.env
✅ 所有必需的环境变量已成功加载
🌐 连接目标: uat-api.clickzetta.com (实例: jnsxwfyr)
📁 工作空间: quick_start.mcp_demo
⚡ 虚拟集群: default
📂 输入目录: /Users/liangmo/yunqidoc/tmp

🎉 环境配置验证成功！


In [5]:
# 🗃️ 第5步：数据库表结构定义

def generate_table_ddl():
    """生成数据库表的 DDL 语句"""
    
    # Raw 表：存储原始的非结构化数据元素
    raw_table_ddl = f"""
CREATE TABLE IF NOT EXISTS {CONFIG['schema']}.{RAW_TABLE_NAME} (
    id STRING,                          -- 主键标识符
    record_locator STRING,              -- 记录定位器（文件路径等）
    type STRING,                        -- 元素类型（Title, NarrativeText等）
    record_id STRING,                   -- 记录标识符
    element_id STRING,                  -- 元素唯一标识符（SHA-256或UUID）
    filetype STRING,                    -- 文件类型（PDF, DOCX, MD等）
    file_directory STRING,              -- 文件目录路径
    filename STRING,                    -- 文件名
    last_modified TIMESTAMP,            -- 文件最后修改时间
    languages STRING,                   -- 文档语言（支持多语言列表）
    page_number STRING,                 -- 页码（适用于PDF、DOCX等）
    text STRING,                        -- 提取的文本内容
    embeddings VECTOR({EMBEDDINGS_DIMENSIONS}), -- 向量嵌入数据
    parent_id STRING,                   -- 父元素ID（用于表示层次结构）
    is_continuation BOOLEAN,            -- 是否为前一元素的延续（分块中使用）
    orig_elements STRING,               -- 原始元素的JSON格式
    element_type STRING,                -- 元素类型详细分类
    coordinates STRING,                 -- 元素坐标（JSON格式存储）
    link_texts STRING,                  -- 链接文本
    link_urls STRING,                   -- 链接URL
    email_message_id STRING,            -- 邮件消息ID
    sent_from STRING,                   -- 发件人
    sent_to STRING,                     -- 收件人
    subject STRING,                     -- 主题
    url STRING,                         -- URL
    version STRING,                     -- 版本
    date_created TIMESTAMP,             -- 创建日期
    date_modified TIMESTAMP,            -- 修改日期
    date_processed TIMESTAMP,           -- 处理日期
    text_as_html STRING,                -- HTML格式文本
    emphasized_text_contents STRING,    -- 强调文本内容
    emphasized_text_tags STRING,        -- 强调文本标签
    documents_original_source STRING    -- 文档原始来源
);
"""

    # Silver 表：处理后的清洁数据，用于生产查询
    silver_table_ddl = f"""
CREATE TABLE IF NOT EXISTS {CONFIG['schema']}.{SILVER_TABLE_NAME} (
    id STRING,                          -- 主键标识符
    record_locator STRING,              -- 记录定位器
    type STRING,                        -- 元素类型
    record_id STRING,                   -- 记录标识符
    element_id STRING,                  -- 元素唯一标识符
    filetype STRING,                    -- 文件类型
    file_directory STRING,              -- 文件目录
    filename STRING,                    -- 文件名
    last_modified TIMESTAMP,            -- 最后修改时间
    languages STRING,                   -- 文档语言
    page_number STRING,                 -- 页码
    text STRING,                        -- 文本内容
    embeddings VECTOR({EMBEDDINGS_DIMENSIONS}), -- 向量嵌入（生产就绪）
    parent_id STRING,                   -- 父元素ID
    is_continuation BOOLEAN,            -- 是否为延续元素
    orig_elements STRING,               -- 原始元素JSON
    element_type STRING,                -- 元素类型
    coordinates STRING,                 -- 元素坐标
    link_texts STRING,                  -- 链接文本
    link_urls STRING,                   -- 链接URL
    email_message_id STRING,            -- 邮件ID
    sent_from STRING,                   -- 发件人
    sent_to STRING,                     -- 收件人
    subject STRING,                     -- 主题
    url STRING,                         -- URL
    version STRING,                     -- 版本
    date_created TIMESTAMP,             -- 创建日期
    date_modified TIMESTAMP,            -- 修改日期
    date_processed TIMESTAMP,           -- 处理日期
    text_as_html STRING,                -- HTML文本
    emphasized_text_contents STRING,    -- 强调内容
    emphasized_text_tags STRING,        -- 强调标签
    documents_source STRING,            -- 文档来源（清理后）
    
    -- 创建倒排索引以支持全文搜索
    INDEX {INDEX_AND_TABLE_PREFIX}inverted_text_index_yunqi_cn (text) 
        INVERTED PROPERTIES('analyzer'='unicode'),
    
    -- 创建向量索引以支持语义搜索
    INDEX {INDEX_AND_TABLE_PREFIX}embeddings_vec_index_yunqi_cn(embeddings) 
        USING VECTOR PROPERTIES (
            "scalar.type" = "f32",
            "distance.function" = "cosine_distance"
        )
);
"""

    # 数据转换SQL：从Raw表清洗数据到Silver表
    transformation_sql = f"""
INSERT OVERWRITE {CONFIG['schema']}.{SILVER_TABLE_NAME}
SELECT 
    id, record_locator, type, record_id, element_id, filetype, 
    file_directory, filename, last_modified, languages, page_number, 
    text, 
    CAST(embeddings AS VECTOR({EMBEDDINGS_DIMENSIONS})) AS embeddings,
    parent_id, is_continuation, orig_elements, element_type, coordinates,
    link_texts, link_urls, email_message_id, sent_from, sent_to, 
    subject, url, version, date_created, date_modified, date_processed,
    text_as_html, emphasized_text_contents, emphasized_text_tags,
    "{DOCUMENTS_SOURCE}" as documents_source
FROM {CONFIG['schema']}.{RAW_TABLE_NAME};
"""

    return raw_table_ddl, silver_table_ddl, transformation_sql

# 生成DDL
RAW_DDL, SILVER_DDL, TRANSFORM_SQL = generate_table_ddl()

print("✅ 数据库表结构定义完成:")
print(f"📋 Raw表: {RAW_TABLE_NAME}")
print(f"✨ Silver表: {SILVER_TABLE_NAME}")
print(f"🔍 索引: 全文搜索 + 向量相似度搜索")
print(f"📏 向量维度: {EMBEDDINGS_DIMENSIONS}")

✅ 数据库表结构定义完成:
📋 Raw表: dashscope_v4_1024_2048_20250611_yunqi_raw_elements
✨ Silver表: dashscope_v4_1024_2048_20250611_yunqi_elements
🔍 索引: 全文搜索 + 向量相似度搜索
📏 向量维度: 1024


In [6]:
# 🔌 第6步：数据库连接管理

from clickzetta.connector import connect
import pandas as pd

class ClickZettaConnectionManager:
    """ClickZetta 数据库连接管理器"""
    
    def __init__(self, config):
        self.config = config
        self._connection = None
    
    def get_connection(self):
        """获取数据库连接"""
        if self._connection is None:
            try:
                self._connection = connect(
                    password=self.config['password'],
                    username=self.config['username'],
                    service=self.config['service'],
                    instance=self.config['instance'],
                    workspace=self.config['workspace'],
                    schema=self.config['schema'],
                    vcluster=self.config['vcluster']
                )
                print("✅ ClickZetta 数据库连接建立成功")
            except Exception as e:
                print(f"❌ 数据库连接失败: {e}")
                raise
        return self._connection
    
    def execute_sql(self, sql_statement: str, description: str = ""):
        """执行SQL语句并返回结果"""
        try:
            with self.get_connection().cursor() as cursor:
                cursor.execute(sql_statement)
                results = cursor.fetchall()
                if description:
                    print(f"✅ {description} - 完成")
                return results
        except Exception as e:
            print(f"❌ SQL执行失败 {f'({description})' if description else ''}: {e}")
            raise
    
    def execute_sql_to_dataframe(self, sql_statement: str, description: str = "") -> pd.DataFrame:
        """执行SQL并返回DataFrame"""
        try:
            with self.get_connection().cursor() as cursor:
                cursor.execute(sql_statement)
                results = cursor.fetchall()
                columns = [desc[0] for desc in cursor.description]
                df = pd.DataFrame(results, columns=columns)
                if description:
                    print(f"✅ {description} - 返回 {len(df)} 行数据")
                return df
        except Exception as e:
            print(f"❌ SQL查询失败 {f'({description})' if description else ''}: {e}")
            raise

# 创建连接管理器
db_manager = ClickZettaConnectionManager(CONFIG)
conn = db_manager.get_connection()

print(f"🌐 已连接到 ClickZetta: {CONFIG['service']}")
print(f"📁 工作空间: {CONFIG['workspace']}.{CONFIG['schema']}")
print(f"⚡ 虚拟集群: {CONFIG['vcluster']}")

✅ ClickZetta 数据库连接建立成功
🌐 已连接到 ClickZetta: uat-api.clickzetta.com
📁 工作空间: quick_start.mcp_demo
⚡ 虚拟集群: default


In [7]:
# 🗂️ 第7步：表管理操作

def manage_tables():
    """管理数据库表的创建和删除"""
    
    if DROP_TABLES_BEFORE_RUN:
        print("🗑️ 删除现有表...")
        # 删除现有表
        db_manager.execute_sql(
            f"DROP TABLE IF EXISTS {CONFIG['schema']}.{RAW_TABLE_NAME}",
            f"删除Raw表 {RAW_TABLE_NAME}"
        )
        db_manager.execute_sql(
            f"DROP TABLE IF EXISTS {CONFIG['schema']}.{SILVER_TABLE_NAME}",
            f"删除Silver表 {SILVER_TABLE_NAME}"
        )
    
    print("🏗️ 创建数据库表...")
    # 创建表
    db_manager.execute_sql(RAW_DDL, f"创建Raw表 {RAW_TABLE_NAME}")
    db_manager.execute_sql(SILVER_DDL, f"创建Silver表 {SILVER_TABLE_NAME}")
    
    print("✅ 表管理操作完成")
    print(f"📋 Raw表: {CONFIG['schema']}.{RAW_TABLE_NAME}")
    print(f"✨ Silver表: {CONFIG['schema']}.{SILVER_TABLE_NAME}")

# 执行表管理
manage_tables()

🗑️ 删除现有表...
✅ 删除Raw表 dashscope_v4_1024_2048_20250611_yunqi_raw_elements - 完成
✅ 删除Silver表 dashscope_v4_1024_2048_20250611_yunqi_elements - 完成
🏗️ 创建数据库表...
✅ 创建Raw表 dashscope_v4_1024_2048_20250611_yunqi_raw_elements - 完成
✅ 创建Silver表 dashscope_v4_1024_2048_20250611_yunqi_elements - 完成
✅ 表管理操作完成
📋 Raw表: mcp_demo.dashscope_v4_1024_2048_20250611_yunqi_raw_elements
✨ Silver表: mcp_demo.dashscope_v4_1024_2048_20250611_yunqi_elements


## 🚀 ETL 数据处理管道

### 数据处理流程
1. **文档摄取**: 扫描本地Markdown文件
2. **内容解析**: 使用unstructured.io提取文档结构  
3. **智能分块**: 按语义边界分割文本
4. **向量嵌入**: 使用DashScope生成1024维向量
5. **数据上传**: 存储到ClickZetta Raw表

In [8]:
# 📦 导入ETL Pipeline依赖

# Core pipeline components
from unstructured_ingest.interfaces import ProcessorConfig
from unstructured_ingest.pipeline.pipeline import Pipeline

# Processing components
from unstructured_ingest.processes.chunker import ChunkerConfig
from unstructured_ingest.processes.embedder import EmbedderConfig
from unstructured_ingest.processes.partitioner import PartitionerConfig

# Local file connector
from unstructured_ingest.processes.connectors.local import (
    LocalIndexerConfig,
    LocalDownloaderConfig,
    LocalConnectionConfig
)

# ClickZetta connector
from unstructured_ingest.processes.connectors.sql.clickzetta import (
    ClickzettaConnectionConfig,
    ClickzettaAccessConfig,
    ClickzettaUploadStagerConfig,
    ClickzettaUploaderConfig
)

print("✅ ETL Pipeline 依赖导入完成")
print("🔗 支持的连接器: Local Files → ClickZetta Lakehouse")
print("🧠 支持的嵌入器: DashScope text-embedding-v4")
print("⚙️  支持的处理器: Chunking, Partitioning, Vector Embedding")

✅ ETL Pipeline 依赖导入完成
🔗 支持的连接器: Local Files → ClickZetta Lakehouse
🧠 支持的嵌入器: DashScope text-embedding-v4
⚙️  支持的处理器: Chunking, Partitioning, Vector Embedding


In [9]:
# 🚀 第8步：构建和执行ETL Pipeline

def create_etl_pipeline():
    """创建并配置ETL数据处理管道"""
    
    # 确定处理模式和并发设置
    num_processes = 1  # 调试模式使用单进程
    file_pattern = "*.md"  # 只处理根目录的MD文件，便于调试
    
    print(f"🔍 创建ETL Pipeline...")
    print(f"📂 输入目录: {CONFIG['input_dir']}")
    print(f"📄 文件模式: {file_pattern}")
    print(f"⚙️  并发进程: {num_processes}")
    print(f"🔧 调试模式: {'开启' if ENABLE_DEBUG_MODE else '关闭'}")
    
    try:
        pipeline = Pipeline.from_configs(
            # 处理器配置
            context=ProcessorConfig(
                verbose=ENABLE_DEBUG_MODE,
                tqdm=True,  # 显示进度条
                num_processes=num_processes,
            ),
            
            # 源数据配置 - 本地文件（限制文件数量便于调试）
            indexer_config=LocalIndexerConfig(
                input_path=CONFIG['input_dir'],
                file_glob=file_pattern,
                recursive=False  # 不递归，只处理根目录文件
            ),
            downloader_config=LocalDownloaderConfig(),
            source_connection_config=LocalConnectionConfig(),
            
            # 文档解析配置
            partitioner_config=PartitionerConfig(
                partition_by_api=False,  # 使用本地解析，不调用API
                strategy="hi_res",       # 高分辨率解析策略
                additional_partition_args={
                    "split_pdf_page": True,
                    "split_pdf_allow_failed": True,
                    "split_pdf_concurrency_level": 1
                }
            ),
            
            # 文本分块配置
            chunker_config=ChunkerConfig(
                chunking_strategy="by_title",           # 按标题分块
                chunk_max_characters=CHUNK_MAX_CHARACTERS,
                chunk_overlap=CHUNK_OVERLAP,
                chunk_combine_text_under_n_chars=CHUNK_COMBINE_THRESHOLD,
            ),
            
            # 向量嵌入配置
            embedder_config=EmbedderConfig(
                embedding_provider=EMBEDDING_PROVIDER,
                embedding_model_name=EMBEDDING_MODEL_NAME,
                embedding_api_key=CONFIG['api_key'],
            ),
            
            # 目标数据库配置 - ClickZetta
            destination_connection_config=ClickzettaConnectionConfig(
                access_config=ClickzettaAccessConfig(password=CONFIG['password']),
                username=CONFIG['username'],
                service=CONFIG['service'],
                instance=CONFIG['instance'],
                workspace=CONFIG['workspace'],
                schema=CONFIG['schema'],
                vcluster=CONFIG['vcluster'],
            ),
            stager_config=ClickzettaUploadStagerConfig(),
            uploader_config=ClickzettaUploaderConfig(
                table_name=RAW_TABLE_NAME,
                documents_original_source=DOCUMENTS_SOURCE,
                batch_size=100  # 小批次便于调试
            ),
        )
        
        print("✅ ETL Pipeline 创建成功")
        return pipeline
        
    except Exception as e:
        print(f"❌ ETL Pipeline 创建失败: {e}")
        raise

def run_etl_pipeline():
    """执行ETL数据处理管道"""
    
    print("\n🚀 开始执行ETL Pipeline（调试模式）...")
    print("=" * 60)
    
    try:
        # 创建并运行pipeline
        pipeline = create_etl_pipeline()
        
        # 执行数据处理
        print("⏳ 正在处理文档...")
        pipeline.run()
        
        print("=" * 60)
        print("🎉 ETL Pipeline 执行成功！")
        print(f"📊 数据已写入表: {CONFIG['schema']}.{RAW_TABLE_NAME}")
        
        # 验证数据
        row_count = db_manager.execute_sql(
            f"SELECT COUNT(*) FROM {CONFIG['schema']}.{RAW_TABLE_NAME}",
            "查询Raw表行数"
        )[0][0]
        
        print(f"✅ 共处理了 {row_count} 条记录")
        
        # 检查数据结构
        sample_data = db_manager.execute_sql_to_dataframe(
            f"SELECT * FROM {CONFIG['schema']}.{RAW_TABLE_NAME} LIMIT 1",
            "获取样本数据"
        )
        
        print(f"📋 表结构验证:")
        print(f"  • 实际列数: {len(sample_data.columns)}")
        print(f"  • 期望列数: 33")
        print(f"  • 列匹配: {'✅' if len(sample_data.columns) == 33 else '❌'}")
        
    except Exception as e:
        print(f"❌ ETL Pipeline 执行失败: {e}")
        import traceback
        traceback.print_exc()
        raise

# 执行ETL流程
run_etl_pipeline()

2025-09-22 11:22:55,683 MainProcess INFO     created indexer with configs: {"input_path":"/Users/liangmo/yunqidoc/tmp","recursive":false}, connection configs: {"access_config":"**********"}
2025-09-22 11:22:55,684 MainProcess INFO     Created download with configs: {"download_dir":null}, connection configs: {"access_config":"**********"}
2025-09-22 11:22:55,684 MainProcess INFO     created partition with configs: {"strategy":"hi_res","ocr_languages":null,"encoding":null,"additional_partition_args":{"split_pdf_page":true,"split_pdf_allow_failed":true,"split_pdf_concurrency_level":1},"skip_infer_table_types":null,"fields_include":["element_id","text","type","metadata","embeddings"],"flatten_metadata":false,"metadata_exclude":[],"element_exclude":[],"metadata_include":[],"partition_endpoint":"https://api.unstructuredapp.io/general/v0/general","partition_by_api":false,"api_timeout_ms":null,"api_key":null,"hi_res_model_name":null,"raise_unsupported_filetype":false}
2025-09-22 11:22:55,684 M


🚀 开始执行ETL Pipeline（调试模式）...
🔍 创建ETL Pipeline...
📂 输入目录: /Users/liangmo/yunqidoc/tmp
📄 文件模式: *.md
⚙️  并发进程: 1
🔧 调试模式: 关闭
✅ ETL Pipeline 创建成功
⏳ 正在处理文档...


2025-09-22 11:22:56,028 MainProcess INFO     running local pipeline: indexer (LocalIndexer) -> download (LocalDownloader) -> partition (hi_res) -> chunk (by_title) -> embed (dashscope) -> upload_stage (ClickzettaUploadStager) -> upload (ClickzettaUploader) with configs: {"reprocess":false,"verbose":false,"tqdm":true,"work_dir":"/Users/liangmo/.cache/unstructured/ingest/pipeline","num_processes":1,"max_connections":null,"raise_on_error":false,"disable_parallelism":false,"preserve_downloads":false,"download_only":false,"re_download":false,"uncompress":false,"iter_delete":false,"delete_cache":false,"otel_endpoint":null,"status":{}}
2025-09-22 11:22:56,035 MainProcess INFO     calling DownloadStep with 1 docs
2025-09-22 11:22:56,036 MainProcess INFO     processing content async
2025-09-22 11:22:56,036 MainProcess WARNING  async code being run in dedicated thread pool to not conflict with existing event loop: <_UnixSelectorEventLoop running=True closed=False debug=False>
2025-09-22 11:22:56

🎉 ETL Pipeline 执行成功！
📊 数据已写入表: mcp_demo.dashscope_v4_1024_2048_20250611_yunqi_raw_elements
✅ 查询Raw表行数 - 完成
✅ 共处理了 4 条记录
✅ 获取样本数据 - 返回 1 行数据
📋 表结构验证:
  • 实际列数: 33
  • 期望列数: 33
  • 列匹配: ✅


## 🔄 数据转换和Silver表构建

将Raw表中的原始数据清洗转换到生产就绪的Silver表中，添加标准化的数据源标识。

In [10]:
# 🔄 第9步：数据转换到Silver表

def check_table_status():
    """检查表的状态和结构"""
    
    print("🔍 检查表状态...")
    
    try:
        # 检查Raw表
        raw_exists = db_manager.execute_sql_to_dataframe(
            f"SHOW TABLES LIKE '{RAW_TABLE_NAME}'",
            "检查Raw表是否存在"
        )
        print(f"📋 Raw表 '{RAW_TABLE_NAME}': {'存在' if len(raw_exists) > 0 else '不存在'}")
        
        if len(raw_exists) > 0:
            raw_count = db_manager.execute_sql(
                f"SELECT COUNT(*) FROM {CONFIG['schema']}.{RAW_TABLE_NAME}"
            )[0][0]
            print(f"  • Raw表记录数: {raw_count}")
        
        # 检查Silver表
        silver_exists = db_manager.execute_sql_to_dataframe(
            f"SHOW TABLES LIKE '{SILVER_TABLE_NAME}'", 
            "检查Silver表是否存在"
        )
        print(f"✨ Silver表 '{SILVER_TABLE_NAME}': {'存在' if len(silver_exists) > 0 else '不存在'}")
        
        if len(silver_exists) > 0:
            try:
                silver_count = db_manager.execute_sql(
                    f"SELECT COUNT(*) FROM {CONFIG['schema']}.{SILVER_TABLE_NAME}"
                )[0][0]
                print(f"  • Silver表记录数: {silver_count}")
                
                # 检查Silver表结构
                silver_schema = db_manager.execute_sql_to_dataframe(
                    f"DESCRIBE {CONFIG['schema']}.{SILVER_TABLE_NAME}",
                    "获取Silver表结构"
                )
                print(f"  • Silver表列数: {len(silver_schema)}")
                
            except Exception as e:
                print(f"  ❌ Silver表查询失败: {e}")
                return False
        
        return len(raw_exists) > 0 and len(silver_exists) > 0
        
    except Exception as e:
        print(f"❌ 表状态检查失败: {e}")
        return False

def transform_to_silver_table():
    """将Raw表数据转换到Silver表"""
    
    print("🔄 开始数据转换...")
    
    # 首先检查表状态
    if not check_table_status():
        print("❌ 表状态检查失败，跳过数据转换")
        return
    
    print(f"📥 源表: {CONFIG['schema']}.{RAW_TABLE_NAME}")
    print(f"📤 目标表: {CONFIG['schema']}.{SILVER_TABLE_NAME}")
    
    try:
        # 检查Raw表是否有数据
        raw_count = db_manager.execute_sql(
            f"SELECT COUNT(*) FROM {CONFIG['schema']}.{RAW_TABLE_NAME}"
        )[0][0]
        
        if raw_count == 0:
            print("⚠️  Raw表中没有数据，跳过转换")
            return
        
        print(f"📊 Raw表中有 {raw_count} 条记录，开始转换...")
        
        # 执行数据转换
        db_manager.execute_sql(TRANSFORM_SQL, "数据转换到Silver表")
        
        # 验证转换结果
        silver_count = db_manager.execute_sql(
            f"SELECT COUNT(*) FROM {CONFIG['schema']}.{SILVER_TABLE_NAME}"
        )[0][0]
        
        print(f"✅ 数据转换完成")
        print(f"📊 Raw表记录数: {raw_count}")
        print(f"✨ Silver表记录数: {silver_count}")
        
        if raw_count == silver_count:
            print("🎯 数据转换验证通过！")
        else:
            print("⚠️  记录数不匹配，请检查转换逻辑")
            
    except Exception as e:
        print(f"❌ 数据转换失败: {e}")
        import traceback
        traceback.print_exc()

# 执行数据转换
transform_to_silver_table()

🔄 开始数据转换...
🔍 检查表状态...
✅ 检查Raw表是否存在 - 返回 1 行数据
📋 Raw表 'dashscope_v4_1024_2048_20250611_yunqi_raw_elements': 存在
  • Raw表记录数: 4
✅ 检查Silver表是否存在 - 返回 1 行数据
✨ Silver表 'dashscope_v4_1024_2048_20250611_yunqi_elements': 存在
  • Silver表记录数: 0
✅ 获取Silver表结构 - 返回 33 行数据
  • Silver表列数: 33
📥 源表: mcp_demo.dashscope_v4_1024_2048_20250611_yunqi_raw_elements
📤 目标表: mcp_demo.dashscope_v4_1024_2048_20250611_yunqi_elements
📊 Raw表中有 4 条记录，开始转换...
✅ 数据转换到Silver表 - 完成
✅ 数据转换完成
📊 Raw表记录数: 4
✨ Silver表记录数: 4
🎯 数据转换验证通过！


## 🔍 语义检索和查询演示

演示如何使用构建好的向量数据库进行语义检索，支持自然语言查询。

In [11]:
# 🔍 第10步：语义检索功能

import dashscope
from dashscope import TextEmbedding
import json

# 配置DashScope API
dashscope.api_key = CONFIG['api_key']

class SemanticSearchEngine:
    """语义搜索引擎"""
    
    def __init__(self, db_manager, config):
        self.db_manager = db_manager
        self.config = config
        self.model_name = EMBEDDING_MODEL_NAME
        self.dimensions = EMBEDDINGS_DIMENSIONS
        self.table_name = SILVER_TABLE_NAME
    
    def check_table_availability(self):
        """检查Silver表是否可用"""
        try:
            # 检查表是否存在
            tables = self.db_manager.execute_sql_to_dataframe(
                f"SHOW TABLES LIKE '{SILVER_TABLE_NAME}'",
                "检查Silver表"
            )
            
            if len(tables) == 0:
                return False, "Silver表不存在"
            
            # 检查表中是否有数据
            count = self.db_manager.execute_sql(
                f"SELECT COUNT(*) FROM {CONFIG['schema']}.{SILVER_TABLE_NAME}"
            )[0][0]
            
            if count == 0:
                return False, "Silver表中没有数据"
            
            return True, f"Silver表可用，包含 {count} 条记录"
            
        except Exception as e:
            return False, f"表检查失败: {e}"
    
    def get_embedding(self, query: str) -> list:
        """使用DashScope获取查询文本的向量嵌入"""
        try:
            response = TextEmbedding.call(
                model=self.model_name,
                input=query
            )
            if response.status_code == 200:
                embedding = response.output['embeddings'][0]['embedding']
                # 验证向量维度
                if len(embedding) != self.dimensions:
                    print(f"⚠️  警告: 嵌入维度 {len(embedding)} 与配置维度 {self.dimensions} 不匹配")
                return embedding
            else:
                raise Exception(f"DashScope API 错误: {response.message}")
        except Exception as e:
            print(f"❌ 获取嵌入向量失败: {e}")
            return [0.0] * self.dimensions
    
    def search_documents(self, query: str, num_results: int = 10, score_threshold: float = None) -> pd.DataFrame:
        """执行语义搜索"""
        print(f"🔍 搜索查询: '{query}'")
        
        # 首先检查表是否可用
        available, message = self.check_table_availability()
        if not available:
            print(f"❌ {message}")
            return pd.DataFrame()
        
        print(f"✅ {message}")
        print(f"📊 返回结果数: {num_results}")
        
        # 获取查询向量
        query_embedding = self.get_embedding(query)
        
        # 构建搜索SQL - 使用更安全的方式
        try:
            sql = f"""
            WITH vector_search_results AS (
                SELECT
                    'vector_similarity' as search_method,
                    record_locator,
                    type,
                    filename,
                    text,
                    orig_elements,
                    cosine_distance(embeddings, CAST({query_embedding} AS VECTOR({self.dimensions}))) AS similarity_score
                FROM {CONFIG['schema']}.{self.table_name}
                WHERE length(text) > 10  -- 过滤空文本
            """
            
            if score_threshold:
                sql += f" AND cosine_distance(embeddings, CAST({query_embedding} AS VECTOR({self.dimensions}))) <= {score_threshold}"
            
            sql += f"""
                ORDER BY similarity_score ASC
                LIMIT {num_results}
            )
            SELECT * FROM vector_search_results
            ORDER BY similarity_score ASC;
            """
            
            results_df = self.db_manager.execute_sql_to_dataframe(
                sql, 
                f"语义搜索 - '{query}'"
            )
            
            print(f"✅ 找到 {len(results_df)} 个相关文档")
            if len(results_df) > 0:
                print(f"🎯 最高相似度分数: {results_df.iloc[0]['similarity_score']:.4f}")
                print(f"📄 最相关文档: {results_df.iloc[0]['filename']}")
            
            return results_df
                
        except Exception as e:
            print(f"❌ 语义搜索失败: {e}")
            # 尝试简单查询作为回退
            try:
                fallback_sql = f"""
                SELECT filename, text, 'fallback' as search_method, 0.0 as similarity_score
                FROM {CONFIG['schema']}.{self.table_name}
                WHERE text LIKE '%{query}%'
                LIMIT {num_results}
                """
                
                print("🔄 尝试文本匹配搜索...")
                results_df = self.db_manager.execute_sql_to_dataframe(
                    fallback_sql,
                    "文本匹配搜索"
                )
                
                if len(results_df) > 0:
                    print(f"✅ 文本搜索找到 {len(results_df)} 个结果")
                
                return results_df
                
            except Exception as e2:
                print(f"❌ 回退搜索也失败: {e2}")
                return pd.DataFrame()
    
    def get_document_chunks(self, filename: str) -> pd.DataFrame:
        """获取指定文档的所有文本块"""
        try:
            sql = f"""
            SELECT record_locator, type, filename, text, orig_elements
            FROM {CONFIG['schema']}.{self.table_name}
            WHERE filename = '{filename}'
            ORDER BY record_locator
            """
            
            return self.db_manager.execute_sql_to_dataframe(
                sql, 
                f"获取文档块 - {filename}"
            )
        except Exception as e:
            print(f"❌ 获取文档块失败: {e}")
            return pd.DataFrame()

# 创建语义搜索引擎
search_engine = SemanticSearchEngine(db_manager, CONFIG)

print("✅ 语义搜索引擎初始化完成")
print(f"🧠 嵌入模型: {EMBEDDING_MODEL_NAME}")
print(f"📏 向量维度: {EMBEDDINGS_DIMENSIONS}")
print(f"🗃️ 搜索表: {SILVER_TABLE_NAME}")

# 检查表可用性
available, message = search_engine.check_table_availability()
print(f"🔍 表状态检查: {message}")

✅ 语义搜索引擎初始化完成
🧠 嵌入模型: text-embedding-v4
📏 向量维度: 1024
🗃️ 搜索表: dashscope_v4_1024_2048_20250611_yunqi_elements
✅ 检查Silver表 - 返回 1 行数据
🔍 表状态检查: Silver表可用，包含 4 条记录


In [12]:
# 🎯 搜索演示：权限管理相关问题

# 检查搜索引擎是否可用
available, message = search_engine.check_table_availability()

if not available:
    print(f"❌ 无法进行搜索演示: {message}")
    print("请确保:")
    print("  1. ETL Pipeline 已成功执行")
    print("  2. 数据已转换到Silver表")
    print("  3. Silver表包含数据")
    
    # 显示空的DataFrame
    search_results = pd.DataFrame()
else:
    print("🔍 开始语义搜索演示...")
    print("=" * 60)
    
    # 定义测试查询
    test_queries = [
        "创建索引的语法是什么？",
        "如何授权用户权限？", 
        "角色管理的最佳实践",
        "数据库安全控制方法"
    ]
    
    # 执行第一个查询作为详细演示
    query = test_queries[0]
    search_results = search_engine.search_documents(query, num_results=5)
    
    print(f"\n📋 搜索结果概览:")
    if not search_results.empty:
        for idx, row in search_results.iterrows():
            score_info = f"(相似度: {row['similarity_score']:.4f})" if 'similarity_score' in row else ""
            print(f"  {idx+1}. 📄 {row['filename']} {score_info}")
            print(f"     📝 预览: {row['text'][:100]}...")
            print()
    else:
        print("  ❌ 未找到相关结果")

# 显示搜索结果DataFrame
search_results

✅ 检查Silver表 - 返回 1 行数据
🔍 开始语义搜索演示...
🔍 搜索查询: '创建索引的语法是什么？'
✅ 检查Silver表 - 返回 1 行数据
✅ Silver表可用，包含 4 条记录
📊 返回结果数: 5
✅ 语义搜索 - '创建索引的语法是什么？' - 返回 4 行数据
✅ 找到 4 个相关文档
🎯 最高相似度分数: 0.6188
📄 最相关文档: access-control.md

📋 搜索结果概览:
  1. 📄 access-control.md (相似度: 0.6188)
     📝 预览: 3.5 权限点

元数据对象的所有操作类型如下：

操作 操作类型 说明 Alter DDL 修改对象的属性 Create DDL 创建对象 Desc DDL 显示单个对象的所有属性 Drop DDL...

  2. 📄 access-control.md (相似度: 0.6298)
     📝 预览: 3.2 元数据对象

Lakehouse中的元数据对象及其父级对象如下表所示：

元数据对象 父级对象 workpace instance share instance network policy ...

  3. 📄 access-control.md (相似度: 0.6780)
     📝 预览: 2.3 预置系统角色

当前系统内预置的角色及权限如下表所示：

角色级别 角色名称 角色代码 具备权限 角色说明 默认授予 实例角色 实例管理员 instance_admin 工作空间——创建工作空...

  4. 📄 access-control.md (相似度: 0.7579)
     📝 预览: 授权

1. 访问控制模型

云器Lakehouse的所有元数据对象均基于访问控制体系进行授权访问。云器Lakehouse支持的访问控制模型包括访问控制列表（ACL，Access Control Li...



,search_method,record_locator,type,filename,text,orig_elements,similarity_score
0,vector_similarity,"{""path"": ""/Users/liangmo/yunqidoc/tmp/access-c...",CompositeElement,access-control.md,3.5 权限点\n\n元数据对象的所有操作类型如下：\n\n操作 操作类型 说明 Alter...,eJztXeuP5MQR/1esjZR8Cbf9fgBCQoAQygGBO5IPHFr182...,0.618788
1,vector_similarity,"{""path"": ""/Users/liangmo/yunqidoc/tmp/access-c...",CompositeElement,access-control.md,3.2 元数据对象\n\nLakehouse中的元数据对象及其父级对象如下表所示：\n\n元...,eJztWsmS20YS/RUEz5a69kV2OGIiJmYuukxYPlkyo1Y1LJ...,0.629823
2,vector_similarity,"{""path"": ""/Users/liangmo/yunqidoc/tmp/access-c...",CompositeElement,access-control.md,2.3 预置系统角色\n\n当前系统内预置的角色及权限如下表所示：\n\n角色级别 角色名称...,eJztXPuPEzkS/lda+flg/H5wCIlbjVa7h245YO+XHRS53T...,0.678022
3,vector_similarity,"{""path"": ""/Users/liangmo/yunqidoc/tmp/access-c...",CompositeElement,access-control.md,授权\n\n1. 访问控制模型\n\n云器Lakehouse的所有元数据对象均基于访问控制体...,eJztXEtvGzkS/iuGzpuY70dumcxlgAC7GGROk8Dgo5hoY1...,0.757923


In [13]:
# 📖 查看最相关文档的详细内容

if 'search_results' in locals() and not search_results.empty:
    # 获取最相关的文档内容
    top_result = search_results.iloc[0]
    print(f"📄 最相关文档: {top_result['filename']}")
    
    if 'similarity_score' in top_result:
        print(f"🎯 相似度分数: {top_result['similarity_score']:.4f}")
    if 'type' in top_result:
        print(f"📝 内容类型: {top_result['type']}")
    if 'search_method' in top_result:
        print(f"🔍 搜索方法: {top_result['search_method']}")
        
    print("=" * 80)
    print("📖 文档内容:")
    print(top_result['text'])
    print("=" * 80)
else:
    print("❌ 没有找到相关文档")
    print("请确保:")
    print("  1. ETL Pipeline 已成功运行")
    print("  2. 数据已存储到Silver表")
    print("  3. 重新运行搜索演示")

📄 最相关文档: access-control.md
🎯 相似度分数: 0.6188
📝 内容类型: CompositeElement
🔍 搜索方法: vector_similarity
📖 文档内容:
3.5 权限点

元数据对象的所有操作类型如下：

操作 操作类型 说明 Alter DDL 修改对象的属性 Create DDL 创建对象 Desc DDL 显示单个对象的所有属性 Drop DDL 删除一个对象 Set DDL 设置参数 Unset DDL 清空一个参数的设置 Show DDL 显示该对象的列表 Truncate DDL 删除表里的数据 Undrop DDL 恢复一个被删除的对象 Use DDL 在一个Session'上下文里使用该对象，比如use database、use vcluster Cancel DDL 取消对象（job对象上应用） Select DML 表数据操作，查询记录 Delete DML 表数据操作，删除记录 Insert DML 表数据操作，插入记录 Insert Overwrite DML 表数据操作，插入并覆盖记录 Merge DML 表数据操作，合并记录 Update DML 表数据操作，更新记录 Replace DML 表数据操作，替换记录 Copy Into DML 将文件里的数据加载到表中 Get DML 文件操作，下载文件 List DML 文件操作，获取文件列表 Put DML 文件操作，上传文件 Remove DML 文件操作，删除文件 Grant DCL 授予权限 Revoke DCL 解除权限

业务对象的所有操作类型如下：

对象 包含操作 脚本（Script） 查看 修改代码 调度配置 【“开发”功能页面】 格式化 保存 发布 运行 回滚 （版本） 任务和实例（task） 查看 调度配置 【“开发”功能页面】 暂停 补数据 启动 下线 重跑 置成功/置失败 恢复 监控规则（monitor rules） 新建监控规则 查看监控规则 查看告警事件 订阅/取消订阅 编辑规则 开启规则 关闭规则 抑制告警事件 关闭告警事件 通知策略（announce policy） 查看通知历史 查看通知策略 新建通知策略 编辑通知策略 删除通知策略 复制通知策略 新建通知配置 修改通知配置 质量规则（dqc rule） 

In [14]:
# 🧠 知识库增强：添加自定义知识

def add_custom_knowledge():
    """向知识库添加自定义知识条目"""
    
    # 首先检查Silver表是否可用
    available, message = search_engine.check_table_availability()
    if not available:
        print(f"❌ 无法添加自定义知识: {message}")
        print("请先确保Silver表已创建并包含数据")
        return
    
    # 定义要添加的知识
    custom_knowledge = """
    ClickZetta、云器、Singdata之间的关系:
    
    ClickZetta是云器、Singdata的技术品牌。云器是ClickZetta的中国商业公司，
    Singdata是ClickZetta的海外商业公司。所以在不区分商业和技术的时候，
    三者是等同的。另外云器经常也会用到Yunqi的汉语拼音名称。
    
    关键信息：
    • ClickZetta: 技术品牌名称
    • 云器(Yunqi): 中国商业公司
    • Singdata: 海外商业公司  
    • 三者在技术层面等同
    """
    
    print("🧠 正在添加自定义知识到向量数据库...")
    
    try:
        # 获取知识文本的向量嵌入
        knowledge_embedding = search_engine.get_embedding(custom_knowledge)
        
        # 构建插入SQL
        insert_sql = f"""
        INSERT INTO {CONFIG['schema']}.{SILVER_TABLE_NAME} (
            id, type, record_id, element_id, filetype, last_modified, 
            languages, text, embeddings, date_created, date_modified, 
            date_processed, documents_source
        ) VALUES (
            uuid(), 
            'UserInput', 
            uuid(), 
            uuid(), 
            'text', 
            CURRENT_TIMESTAMP, 
            '["zh-cn"]',
            '{custom_knowledge.replace("'", "''")}',  -- 转义单引号
            CAST('{knowledge_embedding}' AS VECTOR({EMBEDDINGS_DIMENSIONS})), 
            CURRENT_TIMESTAMP, 
            CURRENT_TIMESTAMP, 
            CURRENT_TIMESTAMP,
            'custom_knowledge'
        );
        """
        
        # 执行插入
        db_manager.execute_sql(insert_sql, "添加自定义知识")
        
        print("✅ 自定义知识添加成功！")
        print("📝 添加的知识内容:")
        print("-" * 60)
        print(custom_knowledge)
        print("-" * 60)
        
        # 验证知识是否可搜索
        print("\n🔍 验证搜索 - 查询'ClickZetta和云器的关系':")
        test_results = search_engine.search_documents("ClickZetta和云器的关系", num_results=3)
        
        if not test_results.empty:
            print(f"✅ 找到 {len(test_results)} 个相关结果")
            if 'similarity_score' in test_results.columns:
                print(f"🎯 最高相似度: {test_results.iloc[0]['similarity_score']:.4f}")
        
    except Exception as e:
        print(f"❌ 添加自定义知识失败: {e}")
        import traceback
        traceback.print_exc()

# 执行知识添加（如果Silver表可用）
add_custom_knowledge()

✅ 检查Silver表 - 返回 1 行数据
🧠 正在添加自定义知识到向量数据库...
✅ 添加自定义知识 - 完成
✅ 自定义知识添加成功！
📝 添加的知识内容:
------------------------------------------------------------

    ClickZetta、云器、Singdata之间的关系:

    ClickZetta是云器、Singdata的技术品牌。云器是ClickZetta的中国商业公司，
    Singdata是ClickZetta的海外商业公司。所以在不区分商业和技术的时候，
    三者是等同的。另外云器经常也会用到Yunqi的汉语拼音名称。

    关键信息：
    • ClickZetta: 技术品牌名称
    • 云器(Yunqi): 中国商业公司
    • Singdata: 海外商业公司  
    • 三者在技术层面等同
    
------------------------------------------------------------

🔍 验证搜索 - 查询'ClickZetta和云器的关系':
🔍 搜索查询: 'ClickZetta和云器的关系'
✅ 检查Silver表 - 返回 1 行数据
✅ Silver表可用，包含 5 条记录
📊 返回结果数: 3
✅ 语义搜索 - 'ClickZetta和云器的关系' - 返回 3 行数据
✅ 找到 3 个相关文档
🎯 最高相似度分数: 0.2101
📄 最相关文档: None
✅ 找到 3 个相关结果
🎯 最高相似度: 0.2101


In [15]:
# 🎉 总结和后续步骤

print("🎉 ClickZetta RAG 知识库构建完成！")
print("=" * 80)

# 获取最终统计信息
try:
    total_records = db_manager.execute_sql(
        f"SELECT COUNT(*) FROM {CONFIG['schema']}.{SILVER_TABLE_NAME}"
    )[0][0]
    
    unique_files = db_manager.execute_sql(
        f"SELECT COUNT(DISTINCT filename) FROM {CONFIG['schema']}.{SILVER_TABLE_NAME}"
    )[0][0]
    
    avg_text_length = db_manager.execute_sql(
        f"SELECT AVG(LENGTH(text)) FROM {CONFIG['schema']}.{SILVER_TABLE_NAME}"
    )[0][0]
    
    print(f"📊 知识库统计:")
    print(f"  • 总记录数: {total_records:,}")
    print(f"  • 唯一文件数: {unique_files:,}")
    print(f"  • 平均文本长度: {avg_text_length:.0f} 字符")
    print(f"  • 向量维度: {EMBEDDINGS_DIMENSIONS}")
    print(f"  • 嵌入模型: {EMBEDDING_MODEL_NAME}")
    
    print(f"\n🗃️ 数据表信息:")
    print(f"  • Raw表: {CONFIG['schema']}.{RAW_TABLE_NAME}")
    print(f"  • Silver表: {CONFIG['schema']}.{SILVER_TABLE_NAME}")
    
    print(f"\n🔍 搜索能力:")
    print(f"  • 向量相似度搜索 (余弦距离)")
    print(f"  • 全文检索 (倒排索引)")
    print(f"  • 自然语言查询支持")
    
except Exception as e:
    print(f"⚠️  统计信息获取失败: {e}")

print("\n🚀 后续步骤建议:")
print("1. 🔧 调整文本分块参数以优化检索效果")
print("2. 📈 监控向量搜索性能并调优索引")
print("3. 🧪 测试不同类型的查询以验证效果")
print("4. 🔄 设置定期数据更新和增量处理")
print("5. 🛡️  配置访问权限和数据安全策略")

print(f"\n✨ 可以开始使用语义搜索功能了！")
print(f"例如：search_engine.search_documents('你的查询')")
print("=" * 80)

🎉 ClickZetta RAG 知识库构建完成！
📊 知识库统计:
  • 总记录数: 5
  • 唯一文件数: 1
  • 平均文本长度: 1309 字符
  • 向量维度: 1024
  • 嵌入模型: text-embedding-v4

🗃️ 数据表信息:
  • Raw表: mcp_demo.dashscope_v4_1024_2048_20250611_yunqi_raw_elements
  • Silver表: mcp_demo.dashscope_v4_1024_2048_20250611_yunqi_elements

🔍 搜索能力:
  • 向量相似度搜索 (余弦距离)
  • 全文检索 (倒排索引)
  • 自然语言查询支持

🚀 后续步骤建议:
1. 🔧 调整文本分块参数以优化检索效果
2. 📈 监控向量搜索性能并调优索引
3. 🧪 测试不同类型的查询以验证效果
4. 🔄 设置定期数据更新和增量处理
5. 🛡️  配置访问权限和数据安全策略

✨ 可以开始使用语义搜索功能了！
例如：search_engine.search_documents('你的查询')
